# Example 16.7 / 1: Seat Six Families at Five Tables

Companion notebook for **16.7 Dinner Seating Arrangement** in the AMPL lab Documentation, section "Seat Six Families at Five Tables". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Seat Six Families at Five Tables

The model below is the formulation above in AMPL. The source instance has families of sizes $m=(6,8,2,9,13,1)$, tables with capacities $c=(8,8,10,4,9)$ and $k=3$. The data assigns the sets and parameters with `ampl.set[...]` and `ampl.param[...]`; the run solves with HiGHS, reads `Seat` with `to_pandas()`, prints the family and table totals, then lowers `max_same_family` to 2 and solves again:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set FAMILIES;  # guest families
set TABLES;  # dinner tables
param members {FAMILIES} integer >= 0;  # guests in each family
param capacity {TABLES} integer >= 0;  # seats at each table
param max_same_family integer >= 0;  # most members of one family per table
var Seat {FAMILIES, TABLES} integer >= 0, <= max_same_family;  # members seated at the table

subject to SeatEveryGuest {f in FAMILIES}:  # every family member gets a seat
    sum {t in TABLES} Seat[f,t] = members[f];
subject to RespectCapacity {t in TABLES}:  # table seats are not exceeded
    sum {f in FAMILIES} Seat[f,t] <= capacity[t];

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
ampl.set["FAMILIES"] = ["F1", "F2", "F3", "F4", "F5", "F6"]
ampl.set["TABLES"] = ["T1", "T2", "T3", "T4", "T5"]
ampl.param["members"] = {"F1": 6, "F2": 8, "F3": 2, "F4": 9, "F5": 13, "F6": 1}
ampl.param["capacity"] = {"T1": 8, "T2": 8, "T3": 10, "T4": 4, "T5": 9}
ampl.param["max_same_family"] = 3

In [5]:
# run.py
ampl.solve(solver="highs", verbose=False)
print(ampl.solve_result)
seat = ampl.var["Seat"].to_pandas()["Seat.val"].round().astype(int).unstack()
print("family totals:", seat.sum(axis=1).to_dict())
print("table totals:", seat.sum(axis=0).to_dict())
print("largest entry:", seat.values.max())
ampl.param["max_same_family"] = 2
ampl.solve(solver="highs", verbose=False)
print("with max_same_family = 2:", ampl.solve_result)

solved
family totals: {'F1': 6, 'F2': 8, 'F3': 2, 'F4': 9, 'F5': 13, 'F6': 1}
table totals: {'T1': 8, 'T2': 8, 'T3': 10, 'T4': 4, 'T5': 9}
largest entry: 3
with max_same_family = 2: infeasible


### Pure AMPL: data.dat and model.run

The same in pure AMPL, with the same `model.mod`:

In [6]:
%%writefile data.dat
set FAMILIES := F1 F2 F3 F4 F5 F6;
set TABLES := T1 T2 T3 T4 T5;
param members := F1 6  F2 8  F3 2  F4 9  F5 13  F6 1;
param capacity := T1 8  T2 8  T3 10  T4 4  T5 9;
param max_same_family := 3;

Writing data.dat


In [7]:
%%ampl_eval
reset;
reset options;
model model.mod;
data data.dat;

In [8]:
# This step fails on purpose, as on the page; the error is printed so the notebook goes on
try:
    ampl.eval(r"""
# model.run
option solver highs, solver_msg 0;
solve;
display solve_result;
display {f in FAMILIES} sum {t in TABLES} Seat[f,t];
display {t in TABLES} sum {f in FAMILIES} Seat[f,t];
let max_same_family := 2;
solve;
display solve_result;
""")
except Exception:
    pass  # AMPL has printed the error above

solve_result = solved

sum{t in TABLES} Seat[f,t] [*] :=
F1   6
F2   8
F3   2
F4   9
F5  13
F6   1
;

sum{f in FAMILIES} Seat[f,t] [*] :=
T1   8
T2   8
T3  10
T4   4
T5   9
;



[WARNING] 
	presolve: constraint SeatEveryGuest['F5'] cannot hold:
		body >= 13 cannot be <= 10; difference = 3


solve_result = infeasible



### Another way in amplpy

The screens explain both results: total capacity $8+8+10+4+9=39$ equals the 39 guests, and the largest family has $13\le3\cdot5=15$, but with $k=2$ it fails $13\le2\cdot5=10$, which is the bound AMPL's presolve reports for `SeatEveryGuest['F5']`.

The run below, with the same model.mod and data.py, tests the other statements of this page. It solves the source instance with `ampl.option["relax_integrality"] = 1`, adds an objective that minimizes the largest family group with `ampl.eval` (amplpy has no method that declares model components), and then loads two small instances after `ampl.eval("reset data;")`:

In [9]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [10]:
# data.py
ampl.set["FAMILIES"] = ["F1", "F2", "F3", "F4", "F5", "F6"]
ampl.set["TABLES"] = ["T1", "T2", "T3", "T4", "T5"]
ampl.param["members"] = {"F1": 6, "F2": 8, "F3": 2, "F4": 9, "F5": 13, "F6": 1}
ampl.param["capacity"] = {"T1": 8, "T2": 8, "T3": 10, "T4": 4, "T5": 9}
ampl.param["max_same_family"] = 3

In [11]:
# run.py
ampl.option["relax_integrality"] = 1
ampl.solve(solver="highs", verbose=False)
values = sorted({round(v, 6) for v in ampl.var["Seat"].to_dict().values()})
print("relaxed:", ampl.solve_result, "Seat values used:", values)

ampl.eval("""
var MaxGroup >= 0;  # largest number of one family at one table
subject to MixFamilies {f in FAMILIES, t in TABLES}:  # every group is at most MaxGroup
    Seat[f,t] <= MaxGroup;
minimize LargestGroup: MaxGroup;  # spread each family as thinly as possible
""")
ampl.solve(solver="highs", verbose=False)
print("relaxed, minimize the largest group:", round(ampl.obj["LargestGroup"].value(), 6))
ampl.option["relax_integrality"] = 0
ampl.solve(solver="highs", verbose=False)
print("integer, minimize the largest group:", round(ampl.obj["LargestGroup"].value(), 6))

small = {
    "one family of 6, tables (3,1,1), k = 3": ({"A": 6}, {"T1": 3, "T2": 1, "T3": 1}, 3),
    "families (2,2), tables (3,1), k = 1": ({"A": 2, "B": 2}, {"T1": 3, "T2": 1}, 1),
}
for label, (members, capacity, k) in small.items():
    ampl.eval("reset data;")
    ampl.set["FAMILIES"] = list(members)
    ampl.set["TABLES"] = list(capacity)
    ampl.param["members"] = members
    ampl.param["capacity"] = capacity
    ampl.param["max_same_family"] = k
    ampl.solve(solver="highs", verbose=False)
    print(f"{label}: {ampl.solve_result}")

relaxed: solved Seat values used: [0, 1, 2, 3]
relaxed, minimize the largest group: 2.6
integer, minimize the largest group: 3.0
one family of 6, tables (3,1,1), k = 3: infeasible
families (2,2), tables (3,1), k = 1: infeasible
